# 05 · EDA avanzado y dataset medido
**Proyecto PLN · Entrega 2** — Pregunta guía: *¿qué tan estable es la agenda temática de la prensa colombiana entre 1980 y 2011 y cuándo cambia?*

Este notebook **entiende el corpus** antes de modelar (el contraste con eventos históricos se deja para el final del notebook 06):

1. Corrección de supuestos de la Entrega 1 (qué se arregla y por qué).
2. Preprocesamiento reproducible (normalización, tokenización, stopwords) y **dataset medido** exportado a parquet/CSV.
3. Diversidad léxica **robusta al tamaño de muestra** (MATTR, Yule's K, Heaps), por año y por fuente.
4. Colocaciones (bigramas/trigramas con NPMI) y términos distintivos (log-odds con prior informativo) por época y fuente.
5. Co-ocurrencias (PPMI) de términos semilla y su cambio de contexto entre épocas.
6. Proxy de entidades (nombres propios) y su peso en el vocabulario.
7. Recurso léxico externo: léxico temático inspirado en el *Comparative Agendas Project* y EuroVoc → categorías iniciales y prevalencia anual.

Ejecución en Colab gratis: ~5 min con el corpus completo (no usa GPU). `FAST = True` hace una corrida reducida de prueba.

Cuadernos previos: `01-corpus` y `02-textometria` (Entrega 1), `03-eda-agenda` y `04-divergencia-lexica` (exploración posterior). Este es el 05; el siguiente es `06-agenda-topicos-cambio`.

## 0. Correcciones respecto a la Entrega 1
| Problema en la Entrega 1 | Por qué es un problema | Qué se hace ahora |
|---|---|---|
| Detector "≥2 de 3 señales en el cuartil superior" | Marca ~25 % de los periodos **por construcción**, haya cambio o no | Se descarta. Se usa un **modelo nulo** (permutaciones) que calibra qué es "cambio" |
| Periodos mixtos (bienio hasta 1989, trimestre desde 1990) | La resolución cambia en mitad de la serie y confunde la comparación | Resolución **anual** homogénea (31–32 años); trimestral queda como mejora |
| JS/coseno sin control del tamaño de muestra | Con pocos documentos la divergencia es alta por ruido muestral | Se calcula el **piso de ruido** con permutaciones a igual N |
| Vocabulario top‑5000 dominado por nombres propios | Los cambios reflejan personajes, no temas | Se mide el peso de entidades y se hace análisis **con y sin** ellas |
| Fuente confundida con tiempo (años 80 ≈ solo Semana) | Un "cambio" puede ser un cambio de fuente | Todo se reporta **por fuente** y los tests se **estratifican por fuente** |
| Dedup min-hash con ruido | Duplicados casi idénticos inflan frecuencias | Se reportan; el impacto es ~1 % y se deja como limitación |

## 1. Configuración y carga

In [ ]:
UTILS_SRC = '"""Utilidades compartidas por los notebooks del proyecto."""\n\nimport hashlib\nimport io\nimport re\nimport urllib.request\nfrom pathlib import Path\n\nimport pandas as pd\n\nCORPUS_URL = "https://github.com/CASDAV/proyecto-pln/releases/download/corpus-v1/noticias_colombia.parquet"\nSHA256 = "92b02aa1a74015192d583cede59c3d8bd7794c214028809cc687704255aa2fc7"\nN_DOCS_UNICOS = 91585\n\n\ndef in_colab() -> bool:\n    try:\n        import google.colab  # noqa: F401\n\n        return True\n    except ImportError:\n        return False\n\n\ndef repo_root() -> Path:\n    for d in [Path.cwd(), *Path.cwd().parents]:\n        if (d / "pyproject.toml").exists():\n            return d\n    return Path.cwd()\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef setup(verbose: bool = True) -> None:\n    """Prepara el entorno de ejecución.\n\n    En Colab instala las dependencias que no vienen preinstaladas. En local\n    no instala nada (eso lo resuelve uv o pip). En ambos casos descarga el\n    corpus de stopwords de NLTK si falta.\n    """\n    if in_colab():\n        import subprocess\n        import sys\n\n        paquetes = ["spacy>=3.8", "nltk", "wordcloud", "pyarrow"]\n        if verbose:\n            print("Instalando dependencias en Colab...")\n        subprocess.run(\n            [sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True\n        )\n\n    import nltk\n\n    try:\n        nltk.data.find("corpora/stopwords")\n    except LookupError:\n        nltk.download("stopwords", quiet=not verbose)\n\n\ndef load_corpus_raw(verify: bool = True) -> pd.DataFrame:\n    """Carga el corpus crudo, sin deduplicación ni limpieza.\n\n    Colab: descarga a memoria en cada sesión.\n    Local: descarga una vez a data/ y reutiliza el archivo.\n    """\n    if in_colab():\n        blob = urllib.request.urlopen(CORPUS_URL).read()\n        got = hashlib.sha256(blob).hexdigest()\n        if got != SHA256:\n            raise RuntimeError(f"checksum no coincide: {got}")\n        return pd.read_parquet(io.BytesIO(blob))\n\n    dest = repo_root() / "data" / "noticias_colombia.parquet"\n\n    if not dest.exists():\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        tmp = dest.with_suffix(".part")\n        urllib.request.urlretrieve(CORPUS_URL, tmp)\n        got = _sha256(tmp)\n        if got != SHA256:\n            tmp.unlink()\n            raise RuntimeError(f"checksum no coincide: {got}")\n        tmp.rename(dest)\n    elif verify and _sha256(dest) != SHA256:\n        raise RuntimeError(f"archivo corrupto en {dest}")\n\n    return pd.read_parquet(dest)\n\n\ndef load_corpus(verify: bool = True) -> pd.DataFrame:\n    """Corpus deduplicado (criterio de 01-corpus) y con `fecha` parseada.\n\n    Descarta duplicados por texto normalizado: minúsculas, espacios\n    colapsados y recortados. Verifica el conteo esperado de documentos.\n    """\n    df_raw = load_corpus_raw(verify=verify)\n\n    norm = df_raw["texto"].str.lower().str.replace(r"\\s+", " ", regex=True).str.strip()\n\n    df = df_raw[~norm.duplicated()].reset_index(drop=True).copy()\n    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")\n\n    if len(df) != N_DOCS_UNICOS:\n        raise RuntimeError(f"esperaba {N_DOCS_UNICOS} documentos, hay {len(df)}")\n\n    return df\n\n\ndef results_dir() -> Path:\n    """Carpeta de resultados: /content/results en Colab, results/ del repo en local."""\n    d = (Path("/content") if in_colab() else repo_root()) / "results"\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\nRE_MILES = re.compile(r"(?<!\\d)(\\d{1,3})(?: [\'’] | \\. )(?=\\d{3}(?!\\d))")\nRE_DEC_COMA = re.compile(r"(?<=\\d) , (?=\\d{1,2}(?!\\d))")\nRE_DEC_PUNTO = re.compile(r"(?<=\\d) \\. (?=\\d{1,2}(?!\\d)(?! *:))")\nRE_AMPM = re.compile(r"(?<=\\d)\\s+([ap])\\s*\\.?\\s+m\\b")\nRE_CTRL = re.compile(r"[\\x80-\\x9f]")\n\n\ndef normalizar_texto(s: pd.Series) -> pd.Series:\n    """Reune cifras y horas que la tokenización de origen partió con espacios."""\n    return (\n        s.str.replace(RE_CTRL, " ", regex=True)\n        .str.replace(RE_MILES, r"\\1", regex=True)\n        .str.replace(RE_DEC_COMA, ",", regex=True)\n        .str.replace(RE_DEC_PUNTO, ".", regex=True)\n        .str.replace(RE_AMPM, r" \\1m", regex=True)\n    )\n'

In [ ]:
import os, sys, re, json, time, warnings, random
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 80)

# utils.py (del repo). Si no está junto al notebook, se crea una copia idéntica.
try:
    import utils
except ImportError:
    open("utils.py", "w", encoding="utf-8").write(UTILS_SRC)
    import utils

try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("spanish"))
except Exception as e:
    print("NLTK no disponible, uso lista de respaldo:", e)
    STOP = set("""de la que el en y a los del se las por un para con no una su al lo como más pero sus le ya o este sí porque esta entre cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso ante ellos e esto mí antes algunos qué unos yo otro otras otra él tanto esa estos mucho quienes nada muchos cual poco ella estar estas algunas algo nosotros mi mis tú te ti tu tus ellas nosotras vosotros vosotras os mío mía míos mías tuyo tuya suyo suya nuestro nuestra es son fue ha han ser era está están sido será fueron he hemos había sea fue""".split())
EXTRA_STOP = set("""dijo año años así ayer hoy además según cada tras sino parte mismo mismos misma puede pueden hacer hace hizo dos tres cuatro cinco mil millones pesos ser va van sido ahora aquí allí bien todavía menos mayor primer primera segundo segunda""".split())
STOP |= EXTRA_STOP

RE_TOK = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+|[.?!]")
TERM = {".", "?", "!"}

def tokenizar(texto):
    """Devuelve (tokens_todos, tokens_contenido, entidades).
    tokens_todos: minúsculas, sin puntuación (para colocaciones).
    tokens_contenido: sin stopwords y len>=3.
    entidades: palabras con mayúscula inicial que no abren oración (proxy de nombres propios)."""
    todos, cont, ent = [], [], []
    inicio = True
    for t in RE_TOK.findall(texto):
        if t in TERM:
            inicio = True
            continue
        low = sys.intern(t.lower())
        todos.append(low)
        es_cont = len(low) >= 3 and low not in STOP
        if es_cont:
            cont.append(low)
            if t[0].isupper() and not inicio and not t.isupper():
                ent.append(low)
        inicio = False
    return todos, cont, ent

def savefig(name, results):
    plt.tight_layout()
    plt.savefig(results / name, dpi=130)
    plt.show()

In [ ]:
FAST = False          # True = corrida reducida (para pruebas)
SEED = 42
DIV_TOKENS = 2000 if FAST else 30000     # tokens de muestra por grupo para diversidad léxica
MATTR_W = 100 if FAST else 500
MIN_V = 5 if FAST else 30                # frecuencia mínima para colocaciones
MIN_V_DISTINTIVOS = 5 if FAST else 50
rng = np.random.default_rng(SEED)

In [ ]:
results = utils.results_dir()
df = utils.load_corpus()
df["texto_limpio"] = utils.normalizar_texto(df["texto"])
df["año"] = df["fecha"].dt.year
print(df.shape, "| fechas inválidas:", df["fecha"].isna().sum())
df = df.dropna(subset=["fecha"]).reset_index(drop=True)
df = df[(df["año"] >= 1980) & (df["año"] <= 2011)].reset_index(drop=True)
print("Documentos 1980–2011:", len(df))
print(pd.crosstab(df["año"], df["fuente"]).T.to_string())

In [ ]:
# Muestra para pruebas rápidas
if FAST:
    df = df.sample(min(len(df), 6000), random_state=SEED).reset_index(drop=True)

## 2. Tokenización y dataset medido
Se tokeniza con expresión regular (letras con tilde/ñ), se pasa a minúsculas y se obtienen tres vistas: *todos los tokens* (para colocaciones), *tokens de contenido* (sin stopwords, longitud ≥ 3) y *entidades proxy* (mayúscula inicial a mitad de oración).
No se lematiza aquí: el lematizador de spaCy sobre 32 M de tokens tarda demasiado en Colab gratis; se deja como mejora y se discute su efecto en el notebook 06.

In [ ]:
t0 = time.time()
tok_all, tok_cont, ents = [], [], []
for txt in df["texto_limpio"].values:
    a, c, e = tokenizar(txt)
    tok_all.append(a); tok_cont.append(c); ents.append(e)
print(f"Tokenizado en {time.time()-t0:.0f}s")
df["n_chars"] = df["texto_limpio"].str.len()
df["n_palabras"] = [len(x) for x in tok_all]
df["n_tokens"] = [len(x) for x in tok_cont]
df["n_types"] = [len(set(x)) for x in tok_cont]
df["ttr"] = df["n_types"] / df["n_tokens"].clip(lower=1)
df["n_entidades"] = [len(x) for x in ents]
df["frac_entidades"] = df["n_entidades"] / df["n_tokens"].clip(lower=1)
print(df[["n_chars","n_palabras","n_tokens","n_types","ttr","frac_entidades"]].describe().round(3).to_string())
print("\nTokens totales (todos):", f"{df.n_palabras.sum():,}", "| contenido:", f"{df.n_tokens.sum():,}")

In [ ]:
# Conteos globales y por (año, fuente) sobre tokens de contenido
uni = Counter()
yc = defaultdict(Counter)   # (año, fuente) -> Counter
for t, y, f in zip(tok_cont, df["año"].values, df["fuente"].values):
    yc[(y, f)].update(t)
for c in yc.values(): uni.update(c)
N_CONT = sum(uni.values())
print("Vocabulario (tipos):", f"{len(uni):,}", "| hapax:", f"{sum(1 for v in uni.values() if v==1)/len(uni):.1%}")
ERA = lambda y: "1980-89" if y < 1990 else ("1990-99" if y < 2000 else "2000-11")
era_c = defaultdict(Counter); src_c = defaultdict(Counter); year_c = defaultdict(Counter)
for (y, f), c in yc.items():
    era_c[ERA(y)].update(c); src_c[f].update(c); year_c[y].update(c)
print({k: sum(v.values()) for k, v in era_c.items()})
print({k: sum(v.values()) for k, v in src_c.items()})

## 3. Diversidad léxica robusta al tamaño
El TTR y el tamaño de vocabulario **crecen con el N**, así que comparar años con distinto volumen es engañoso. Se usan:
* **MATTR** (moving-average TTR, ventana fija): estable ante la longitud del texto.
* **Yule's K** (concentración; mayor K = vocabulario más repetitivo).
* **Muestra de igual tamaño** (`DIV_TOKENS` tokens) por grupo, con documentos al azar; si un grupo no alcanza ese N se marca `NaN`.
* **Ley de Heaps** V = K·Nᵝ (β más alto = el vocabulario sigue abriéndose).

In [ ]:
def mattr(tokens, w=500):
    n = len(tokens)
    if n < w: return np.nan
    c = Counter(tokens[:w]); distinct = len(c); s = distinct
    for i in range(w, n):
        a = tokens[i - w]; c[a] -= 1
        if c[a] == 0: del c[a]; distinct -= 1
        b = tokens[i]
        if b not in c: distinct += 1
        c[b] += 1
        s += distinct
    return s / w / (n - w + 1)

def yule_k(tokens):
    c = Counter(tokens); N = sum(c.values())
    if N == 0: return np.nan
    m = Counter(c.values())
    return 1e4 * (sum(f * f * v for f, v in m.items()) - N) / (N * N)

def muestra_tokens(idx, n_target):
    idx = rng.permutation(idx); out = []
    for i in idx:
        out.extend(tok_cont[i])
        if len(out) >= n_target: return out[:n_target]
    return None  # no alcanza

rows = []
for (y, f), g in df.groupby(["año", "fuente"]):
    s = muestra_tokens(g.index.values, DIV_TOKENS)
    if s is None: continue
    rows.append({"año": y, "fuente": f, "mattr": mattr(s, MATTR_W), "yule_k": yule_k(s), "ttr_muestra": len(set(s))/len(s)})
for y, g in df.groupby("año"):
    s = muestra_tokens(g.index.values, DIV_TOKENS)
    if s is None: continue
    rows.append({"año": y, "fuente": "todas", "mattr": mattr(s, MATTR_W), "yule_k": yule_k(s), "ttr_muestra": len(set(s))/len(s)})
div = pd.DataFrame(rows)
print("Grupos con muestra suficiente:", len(div))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for f, g in div.groupby("fuente"):
    ax[0].plot(g["año"], g["mattr"], marker="o", ms=3, label=f)
    ax[1].plot(g["año"], g["yule_k"], marker="o", ms=3, label=f)
ax[0].set_title(f"MATTR (ventana {MATTR_W}, {DIV_TOKENS:,} tokens por grupo)"); ax[1].set_title("Yule's K")
ax[0].legend(); [a.set_xlabel("año") for a in ax]
savefig("fig_diversidad_lexica.png", results)
div.to_csv(results / "diversidad_lexica.csv", index=False)

In [ ]:
# Ley de Heaps (orden aleatorio de documentos) global y por fuente
def heaps(idx, pasos=40):
    idx = rng.permutation(idx); seen = set(); N = 0; pts = []
    total = sum(len(tok_cont[i]) for i in idx); marca = total / pasos; nxt = marca
    for i in idx:
        seen.update(tok_cont[i]); N += len(tok_cont[i])
        if N >= nxt: pts.append((N, len(seen))); nxt += marca
    if len(pts) < 3: return np.nan, np.nan
    x, v = np.log([p[0] for p in pts[2:]]), np.log([p[1] for p in pts[2:]])
    beta, logk = np.polyfit(x, v, 1)
    return np.exp(logk), beta
hp = {"global": heaps(df.index.values)}
for f, g in df.groupby("fuente"): hp[f] = heaps(g.index.values)
print(pd.DataFrame(hp, index=["K", "beta"]).T.round(3).to_string())
print("β≈0,5 (típico de Heaps): el vocabulario sigue abriéndose con el tamaño del corpus (nombres propios, cifras, neologismos); no se puede 'cerrar'.")

## 4. Colocaciones: bigramas y trigramas (NPMI)
NPMI = log(p(ab)/(p(a)p(b))) / −log p(ab) ∈ [−1, 1]. Solo se cuentan pares **adyacentes** donde ambas palabras son de contenido (no stopwords) y tienen frecuencia ≥ `MIN_V`. Los trigramas se construyen solo sobre bigramas con NPMI alto.

In [ ]:
V = {w for w, c in uni.items() if c >= MIN_V}
bi = Counter()
for t in tok_all:
    bi.update([(a, b) for a, b in zip(t, t[1:]) if a in V and b in V])
B = sum(bi.values())
def npmi(a, b, c):
    pab = c / B; pa = uni[a] / N_CONT; pb = uni[b] / N_CONT
    return np.log(pab / (pa * pb)) / (-np.log(pab))
rows = [(a, b, c, npmi(a, b, c)) for (a, b), c in bi.items() if c >= MIN_V]
bg = pd.DataFrame(rows, columns=["w1", "w2", "n", "npmi"]).sort_values("npmi", ascending=False)
bg["bigrama"] = bg.w1 + " " + bg.w2
print("Bigramas con n≥%d: %d" % (MIN_V, len(bg)))
print(bg.head(30)[["bigrama", "n", "npmi"]].round(3).to_string(index=False))
BG = set(zip(bg[bg.npmi >= 0.4].w1, bg[bg.npmi >= 0.4].w2))
tri = Counter()
for t in tok_all:
    tri.update([(a, b, c) for a, b, c in zip(t, t[1:], t[2:]) if (a, b) in BG and (b, c) in BG])
tg = pd.DataFrame([(" ".join(k), v) for k, v in tri.items() if v >= max(3, MIN_V // 2)], columns=["trigrama", "n"]).sort_values("n", ascending=False)
print("\nTrigramas frecuentes (formados por dos colocaciones NPMI≥0.4):"); print(tg.head(20).to_string(index=False))
bg.head(2000).to_csv(results / "bigramas_npmi.csv", index=False)

In [ ]:
# Colocaciones más frecuentes (NPMI≥0.3) por época
BG3 = set(zip(bg[bg.npmi >= 0.3].w1, bg[bg.npmi >= 0.3].w2))
bi_era = defaultdict(Counter)
for t, y in zip(tok_all, df["año"].values):
    bi_era[ERA(y)].update([(a, b) for a, b in zip(t, t[1:]) if (a, b) in BG3])
out = {e: [" ".join(k) + f" ({v})" for k, v in c.most_common(10)] for e, c in sorted(bi_era.items())}
print(pd.DataFrame(out).to_string(index=False))

## 5. Términos distintivos (log-odds con prior informativo de Dirichlet — Monroe et al. 2008)
Frente a la frecuencia bruta o TF-IDF, el z-score de log-odds **penaliza términos raros** y sirve para comparar grupos de distinto tamaño. Se comparan épocas contra el resto y fuentes contra el resto.

In [ ]:
vocab = [w for w, c in uni.items() if c >= MIN_V_DISTINTIVOS]
widx = {w: i for i, w in enumerate(vocab)}
def vec(c):
    v = np.zeros(len(vocab))
    for w, i in widx.items(): v[i] = c.get(w, 0)
    return v
glob = vec(uni); alpha0 = 500.0; alpha = glob / glob.sum() * alpha0
def logodds_z(ci, cj):
    yi, yj = vec(ci), vec(cj); ni, nj = yi.sum(), yj.sum()
    d = np.log((yi + alpha) / (ni + alpha0 - yi - alpha)) - np.log((yj + alpha) / (nj + alpha0 - yj - alpha))
    z = d / np.sqrt(1 / (yi + alpha) + 1 / (yj + alpha))
    return pd.Series(z, index=vocab)
def contra_resto(grupos, k=15):
    out = {}
    for g, c in grupos.items():
        resto = Counter()
        for h, c2 in grupos.items():
            if h != g: resto.update(c2)
        z = logodds_z(c, resto)
        out[g] = list(z.sort_values(ascending=False).head(k).index)
    return pd.DataFrame(out)
print("== Términos distintivos por ÉPOCA =="); print(contra_resto(era_c).to_string(index=False))
print("\n== Términos distintivos por FUENTE =="); print(contra_resto(src_c).to_string(index=False))

## 6. Co-ocurrencias (PPMI a nivel documento) y cambio de contexto
Para términos semilla se calcula PMI de co-aparición en el mismo documento y se comparan los vecinos más asociados entre épocas: es una forma barata de ver si **un mismo concepto cambia de contexto** (por ejemplo "violencia" asociada a guerrilla vs. a delincuencia común).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
top_v = [w for w, c in uni.most_common(20000 if not FAST else 3000)]
cv = CountVectorizer(analyzer=lambda x: x, vocabulary={w: i for i, w in enumerate(top_v)}, binary=True)
Xb = cv.fit_transform(tok_cont).tocsr()
inv = np.array(top_v)
anios = df["año"].values
def vecinos(semilla, mask, k=10, min_co=None):
    if semilla not in cv.vocabulary_: return []
    X = Xb[mask]; D = X.shape[0]; j = cv.vocabulary_[semilla]
    col = X[:, j].toarray().ravel()
    dfs = np.asarray(X.sum(0)).ravel(); co = X.T @ col
    min_co = min_co or (3 if FAST else 20)
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log(co * D / (col.sum() * dfs))
    pmi[(co < min_co) | (dfs < min_co)] = -np.inf; pmi[j] = -np.inf
    return list(inv[np.argsort(-pmi)[:k]])
SEMILLAS = ["violencia", "paz", "economía", "guerrilla", "corrupción", "salud", "empleo", "fútbol"]
eras = {"1980-89": anios < 1990, "1990-99": (anios >= 1990) & (anios < 2000), "2000-11": anios >= 2000}
tab = {s: {e: ", ".join(vecinos(s, m, 8)) for e, m in eras.items()} for s in SEMILLAS if s in cv.vocabulary_}
print(pd.DataFrame(tab).T.to_string())

## 7. Entidades (proxy de nombres propios)
Las entidades dominan el vocabulario de prensa. Si los "cambios temáticos" se debieran a personajes, un análisis temático tendría que separarlos. Se mide su peso y las más frecuentes por época.

In [ ]:
ent_era = defaultdict(Counter)
for e, y in zip(ents, df["año"].values): ent_era[ERA(y)].update(e)
print(pd.DataFrame({k: [f"{w} ({c})" for w, c in v.most_common(12)] for k, v in sorted(ent_era.items())}).to_string(index=False))
ent_year = df.groupby("año").apply(lambda g: g.n_entidades.sum() / g.n_tokens.sum())
print("\nFracción de tokens de contenido que son entidades: media %.1f%%, rango %.1f–%.1f%%" % (100*ent_year.mean(), 100*ent_year.min(), 100*ent_year.max()))
cobertura = []
for e, c in ent_era.items():
    tot = sum(era_c[e].values()); cobertura.append((e, sum(c.values()) / tot))
print(cobertura)
fig, ax = plt.subplots(figsize=(7, 3.5)); ent_year.plot(ax=ax, marker="o", ms=3); ax.set_title("Fracción de tokens que son entidades (proxy)")
savefig("fig_entidades.png", results)

## 8. Recurso léxico externo: léxico temático (CAP / EuroVoc adaptado)
Sin acceso garantizado a recursos descargables en Colab, se construyó un **léxico temático propio** de 13 macro‑categorías inspirado en los *major topics* del Comparative Agendas Project y en los dominios de EuroVoc. Cada categoría es una lista de **prefijos** (raíces) que se expanden contra el vocabulario del corpus.
Sirve a tres fines: (i) **categorías iniciales** por documento para el dataset medido, (ii) una primera medida *model‑free* de agenda (prevalencia = tokens de la categoría por mil tokens de contenido, normalización análoga a la del paper de Caicedo et al.), (iii) validar luego los tópicos no supervisados del notebook 06.
Límites: cobertura incompleta, ambigüedad léxica (p. ej. "partido"), y decisiones de diseño subjetivas → se discuten en las limitaciones.

In [ ]:
LEXICON = {
 "conflicto_armado": r"guerrill|paramilit|secuestr|farc|eln|masacre|terroris|atentad|subversi|insurgen|autodefensa|combate|bombard|desplazad|minas?$",
 "narcotrafico": r"narco|cartel|cartéles|cocaín|coca$|extradici|estupefaci|droga|mafia|sicari",
 "politica_gobierno": r"gobiern|president|congres|senad|ministr|elecci|candidat|liberal|conservador|alcald|gobernador|parlament|reform|constituy|constituci|ley$|leyes|decreto",
 "justicia_crimen": r"juez|jueces|fiscal|tribunal|justicia|sentencia|condena|delito|judicial|cárcel|carcel|policía|policia|homicid|asesinat|captur|denunci",
 "economia_finanzas": r"econom|inflaci|banco|bancari|mercado|inversi|bolsa|dólar|dolar|deuda|crédit|credit|empresa|exportaci|importaci|financier|tasa|impuest|presupuest|fiscal$",
 "empleo_trabajo": r"desempleo|sindicat|trabajador|salari|empleo|huelga|laboral|paro$|obrer",
 "salud": r"salud|hospital|enfermedad|médic|medic|clínica|clinica|vacun|epidemi|sida$|paciente",
 "educacion": r"educaci|colegi|universidad|estudiant|maestr|docente|escolar|profesor|alumn",
 "deporte": r"fútbol|futbol|equipo|gol$|goles|campeonato|ciclis|torneo|jugador|selecci|mundial|vuelta$|tenis|atlet|entrenador",
 "cultura_entretenimiento": r"música|musica|cine$|película|pelicula|artista|libro|teatro|festival|novela|escritor|concierto|televisi|cantante|pintur",
 "ambiente_energia": r"ambient|petról|petrol|energ|ecolog|contamin|agua$|aguas|forestal|ecosistem|eléctric|electric|gas$",
 "internacional": r"venezuel|cuba$|europ|onu$|internacional|extranjer|washington|méxic|mexic|ecuador|panam|estados$|unidos|brasil|perú",
 "infraestructura_vivienda": r"carretera|transporte|tránsito|transito|aeropuerto|metro$|vivienda|construcci|obras?$|vías|vial|puente|acueducto",
}
CATS = list(LEXICON)
cat_of = {}
for c, pat in LEXICON.items():
    rx = re.compile("^(?:" + pat + ")")
    for w in uni:
        if w not in cat_of and rx.search(w): cat_of[w] = c
print("Palabras del corpus cubiertas por el léxico:", len(cat_of), "| tokens cubiertos: %.1f%%" % (100 * sum(uni[w] for w in cat_of) / N_CONT))
for c in CATS:
    ws = sorted([w for w, k in cat_of.items() if k == c], key=lambda w: -uni[w])[:8]
    print(f"  {c:26s}", ws)

In [ ]:
# Categoría inicial por documento + prevalencia por año
cat_idx = {c: i for i, c in enumerate(CATS)}
hits = np.zeros((len(df), len(CATS)), dtype=np.int32)
for i, t in enumerate(tok_cont):
    for w in t:
        c = cat_of.get(w)
        if c is not None: hits[i, cat_idx[c]] += 1
tot_hits = hits.sum(1)
MIN_HITS = 2
best = hits.argmax(1)
df["categoria_inicial"] = np.where(tot_hits >= MIN_HITS, np.array(CATS)[best], "sin_categoria")
df["n_hits_lexico"] = tot_hits
print(df["categoria_inicial"].value_counts(normalize=True).round(3).to_string())
prev = pd.DataFrame(hits, columns=CATS); prev["año"] = df["año"].values; prev["tok"] = df["n_tokens"].values
g = prev.groupby("año")
prev_year = g[CATS].sum().div(g["tok"].sum(), axis=0) * 1000   # por mil tokens de contenido
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(prev_year.div(prev_year.mean()).T.values, aspect="auto", cmap="RdBu_r", vmin=0, vmax=2)
ax.set_yticks(range(len(CATS))); ax.set_yticklabels(CATS)
ax.set_xticks(range(0, len(prev_year), 2)); ax.set_xticklabels(prev_year.index[::2], rotation=45)
ax.set_title("Prevalencia léxica por año (relativa a la media de cada categoría; 1 = promedio)"); plt.colorbar(im)
savefig("fig_lexico_prevalencia.png", results)
prev_year.round(3).to_csv(results / "prevalencia_lexico_por_anio.csv")
# Prevalencia por fuente
prev["fuente"] = df["fuente"].values
gs = prev.groupby("fuente"); prev_src = (gs[CATS].sum().div(gs["tok"].sum(), axis=0) * 1000).round(2)
print("\nPrevalencia por mil tokens, por fuente:"); print(prev_src.T.to_string())

## 9. Dataset medido (parquet/CSV)
Columnas: `id`, `fuente`, `fecha`, `año`, `texto_limpio`, `tokens` (tokens de contenido separados por espacio), longitudes (`n_chars`, `n_palabras`, `n_tokens`, `n_types`, `ttr`), proxy de entidades (`n_entidades`, `frac_entidades`) y `categoria_inicial` (+ `n_hits_lexico`).

In [ ]:
df["tokens"] = [" ".join(t) for t in tok_cont]
cols = ["id", "fuente", "fecha", "año", "texto_limpio", "tokens", "n_chars", "n_palabras", "n_tokens", "n_types", "ttr",
        "n_entidades", "frac_entidades", "categoria_inicial", "n_hits_lexico"]
out = df[cols]
try:
    out.to_parquet(results / "corpus_medido.parquet", index=False)
    print("parquet guardado:", (results / "corpus_medido.parquet").stat().st_size / 1e6, "MB")
except Exception as e:
    print("No se pudo guardar parquet (", e, "); se guarda CSV.gz")
    out.to_csv(results / "corpus_medido.csv.gz", index=False)
out.sample(min(2000, len(out)), random_state=SEED).to_csv(results / "corpus_medido_muestra.csv", index=False)
out.drop(columns=["texto_limpio"]).head(3).T

In [ ]:
# Ejemplo concreto de medición
r = out.iloc[0]
print("id:", r["id"], "|", r["fuente"], "|", r["fecha"].date())
print("TEXTO:", r["texto_limpio"][:400], "...")
print("TOKENS:", r["tokens"][:300], "...")
print({k: r[k] for k in ["n_chars", "n_palabras", "n_tokens", "n_types", "ttr", "n_entidades", "categoria_inicial"]})

In [ ]:
resumen = {
  "n_docs": int(len(df)), "n_tokens_todos": int(df.n_palabras.sum()), "n_tokens_contenido": int(df.n_tokens.sum()),
  "tipos_contenido": int(len(uni)), "frac_hapax": float(sum(1 for v in uni.values() if v == 1) / len(uni)),
  "heaps_beta": {k: float(v[1]) for k, v in hp.items()},
  "frac_entidades_media": float(ent_year.mean()),
  "docs_con_categoria": float((df.categoria_inicial != "sin_categoria").mean()),
  "cobertura_lexico_tokens": float(sum(uni[w] for w in cat_of) / N_CONT),
}
json.dump(resumen, open(results / "resumen_05.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print("Archivos en", results, ":", sorted(p.name for p in results.iterdir()))

## 10. Hallazgos de la corrida de referencia (corpus completo)
* **Corpus**: 91 585 documentos, 31,9 M de tokens (15,5 M de contenido), 266 562 tipos, 42,3 % de hapax. Mediana de 245 palabras por artículo.
* **Diversidad léxica**: el MATTR baja de ~0,80 (años 80–90) a ~0,75 (2000s) y cae a la vez en las tres fuentes hacia 2001 y 2004–05, lo que sugiere cambios de formato en el archivo y no de estilo. Yule's K: Dinero (4,3–5,5) > Semana (3,3–4,6) > El Tiempo (~3–3,5).
* **Heaps**: β = 0,46 (global y El Tiempo), 0,50 (Semana), 0,51 (Dinero): el vocabulario no se satura.
* **Colocaciones y distintivos**: *derechos humanos*, *Naciones Unidas*, *Juan Manuel Santos*, *presidente Álvaro Uribe*; por época, Reagan/Betancur (80), Samper/Gaviria (90), Uribe/FARC/paramilitares/internet (2000+); por fuente, local (El Tiempo), conflicto nacional (Semana), economía (Dinero).
* **Co‑ocurrencias**: el contexto de *salud*, *economía* y *guerrilla* cambia entre épocas (Sisbén/EPS/Fosyga; PIB/recesión/Bernanke; Tirofijo/Caguán/Jojoy). Las vecindades de 1980–89 para *corrupción*, *empleo* y *fútbol* son ruido de pocas co‑ocurrencias.
* **Entidades**: 22 % de los tokens de contenido (12–15 % en los 80, 17–20 % desde 1990; el salto coincide con la llegada de El Tiempo).
* **Léxico temático**: 2 319 palabras (6,8 % de los tokens); narcotráfico máximo en 1987–89, conflicto armado en 2001–02, economía en 1999–2002, justicia/crimen creciente desde 2005. Los años 80 son ruidosos (31–166 documentos por año).
* **Siguiente**: `06-agenda-topicos-cambio.ipynb` (representaciones, tópicos, estabilidad y puntos de cambio).